# Positional Encoding

Ya hemos visto que la arquitectura ***Glimpse*** entrega buenos resultados por sí sola. De ahora en adelante, nos enfocaremos en comprender e implementar técnicas complementarias que, aunque no son 100% obligatorias, tienen un amplio potencial de mejorar la capacidad de representación del modelo. Una de estas técnicas es **Positional Encoding** (PE).

El Positional Encoding es una técnica arquitectónica que permite al modelo comprender la noción del "orden" relativo entre los elementos. Este orden puede entenderse de dos formas distintas:

1. Un orden **espacial**: mide cuál es la posición relativa o distancia **física** entre los elementos (qué tan alejado o cerca se encuentra uno del otro).
2. Un orden **secuencial**: representa el orden jerárquico entre los elementos (cuál va primero y cuál va después) ya sea por algún criterio de prioridad o por dependencia temporal.

En el TSP, el orden espacial se representa mediante lo que se conoce como **PE Espacial**, mientras que para modelar el orden secuencial existen dos variantes: **PE Tradicional** (el clásico) y **PE Cíclico**. Ahondaremos en cada una de estas técnicas a continuación.

## Preparación del entorno

Antes que nada, prepararemos los datos para el entrenamiento de los modelos.

In [1]:
from instances.instances import generate_instances

instances = generate_instances(filename="TSP50.pkl", instance_count=1000, cities=50, seed=42)

from data.generation import generate_train_data
from data.adapters.input.default import DefaultInputAdapter
from data.adapters.output.default import DefaultOutputAdapter

input_config = (DefaultInputAdapter, 50)
output_config = (DefaultOutputAdapter, 50)

generate_train_data(
    instance_file="TSP50.pkl", 
    data_filename="train_data.h5", 
    input_adapter_config=input_config, 
    output_adapter_config=output_config,
    size=5000
)

from data.preprocessing import load_dataset, split_dataset

# Split 80/20
train_file, val_file = split_dataset("train_data.h5", 4000)

train_dataset = load_dataset(train_file)
val_dataset = load_dataset(val_file)

Datos guardados en: /home/oscar/Escritorio/TSP-Framework/data/train_data.h5 (Tamaño: 5000)
Cargando dataset original: /home/oscar/Escritorio/TSP-Framework/data/train_data.h5
Guardando Train puro (4000 muestras) en: train_data_train.h5
Guardando Val puro (1000 muestras) en: train_data_val.h5
Dataset train_data_train.h5 cargado con 4000 muestras.
Dataset train_data_val.h5 cargado con 1000 muestras.


## PE Espacial

### Justificación

Antes de aplicar PE, y en particular su variante espacial, debemos preguntarnos **por qué** y **dónde** sería útil aplicar esta técnica dentro de la arquitectura.

**El por qué**

Nuestro modelo actual (Glimpse) tiene la capacidad de entender la noción de "posición" mediante las coordenadas $(X, Y)$ que pasamos como *features*. ¿Es suficiente? No sabemos exactamente los cálculos internos que aplica el modelo para predecir la siguiente ciudad, sin embargo, podemos intuir de que como mínimo necesita entender la noción de **distancia** entre puntos. Esto ya sea para determinar las ciudades más cercanas a la actual (y por tanto las más prometedoras), o para encontrar agrupaciones (*clusters*) de puntos que sea conveniente visitar primero, o bien para comprender la distancia o costo del *tour* hasta el momento.

Actualmente, esperamos que el modelo deduzca todas estas representaciones mediante un simple listado de coordenadas. Suena poco razonable.

**El dónde**

Actualmente procesamos la posición $(X, Y)$ de las ciudades dentro del *encoder*, dado que corresponde a la información base del problema y no depende del *tour*. De la misma manera, la distancia entre ellas (que puede entenderse como las aristas del grafo de ciudades) también corresponde a información propia de la instancia. Por consiguiente, tiene sentido integrar el PE Espacial dentro del *encoder*, que es donde esperamos que el modelo entienda la representación fundamental del problema.

### Definición matemática

El PE espacial consiste simplemente en **sumar** un *embedding espacial* al *embedding* actual de cada ciudad. La clave está en cómo obtenemos este *embedding espacial*:

$$\begin{aligned} \text{PE}_x^{(j)}(i) &= \left( \sin\left(\frac{x_i}{10000^{2j/(d_{model}/2)}}\right), \cos\left(\frac{x_i}{10000^{2j/(d_{model}/2)}}\right) \right) \\ \text{PE}_y^{(j)}(i) &= \left( \sin\left(\frac{y_i}{10000^{2j/(d_{model}/2)}}\right), \cos\left(\frac{y_i}{10000^{2j/(d_{model}/2)}}\right) \right) \end{aligned}$$

Tenemos dos coordenadas: $x_i$ e $y_i$ para cada ciudad $i$. Necesitamos un embedding capaz de representar ambas coordenadas. Para ello, crearemos dos *mini-embeddings* de tamaño $d_{model}/2$ donde el primero represente la coordenada $x_i$ y el segundo la coordenada $y_i$. Eso es lo que representan ambas ecuaciones. Posteriormente, podremos concatenar ambos embeddings y aplicar la suma directamente, ya que volveremos a la dimensión $d_{model}$ original.

Y bien, ¿cómo generamos cada *embedding*? Si miramos las ecuaciones, notaremos que existe una variable $j$. Dado que cada *mini-embedding* se compone de pares (seno y coseno), la variable $j$ indica el índice de estos pares y llega hasta la mitad del tamaño del *mini-embedding* ($j = [0, 1, ..., d_{model}/4-1]$). Por cada posición $j$ aplicaremos las funciones $\sin$ y $\cos$ de manera intercalada de la siguiente manera:

$$\begin{aligned} \text{PE}_x(i) &= \left[ \sin\left(\frac{x_i}{10000^{0/(d_{model}/2)}}\right), \cos\left(\frac{x_i}{10000^{0/(d_{model}/2)}}\right), \sin\left(\frac{x_i}{10000^{2/(d_{model}/2)}}\right), \cos\left(\frac{x_i}{10000^{2/(d_{model}/2)}}\right), ... \right] \\ \text{PE}_y(i) &= \left[ \sin\left(\frac{y_i}{10000^{0/(d_{model}/2)}}\right), \cos\left(\frac{y_i}{10000^{0/(d_{model}/2)}}\right), \sin\left(\frac{y_i}{10000^{2/(d_{model}/2)}}\right), \cos\left(\frac{y_i}{10000^{2/(d_{model}/2)}}\right), ... \right] \end{aligned}$$

### Implementación y entrenamiento

La implementación completa de la arquitectura puede visualizarse en el archivo correspondiente.

In [2]:
from models.spatial_pe import TSPTransformer

# Parámetros del modelo
embed_dim = 64
num_heads = 4
num_encoder_layers = 2
num_glimpses = 2
dropout = 0.1

# Crear modelo
spe_model = TSPTransformer(
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_encoder_layers=num_encoder_layers,
    num_glimpses=num_glimpses,
    dropout_rate=dropout
)

from training.sl import sl_train, LRConfig
from training.metrics import CrossEntropyLoss, Accuracy

spe_model = sl_train(
    model=spe_model,
    epochs=10,
    train_set=train_dataset,
    val_set=val_dataset,
    batch_size=64,
    lr_config=LRConfig(value=1e-4),
    loss_fn=CrossEntropyLoss(),
    metrics=[Accuracy()],
    metrics_filename="metrics_spatial_pe.txt"
)

** Usando dispositivo: cpu

Epoch 1/10
    Train CrossEntropy: 3.0205 | Val CrossEntropy: 2.9961
    Accuracy: 14.70%
Epoch 2/10
    Train CrossEntropy: 3.0211 | Val CrossEntropy: 2.9958
    Accuracy: 15.40%
Epoch 3/10
    Train CrossEntropy: 3.0189 | Val CrossEntropy: 2.9958
    Accuracy: 15.50%
Epoch 4/10
    Train CrossEntropy: 3.0178 | Val CrossEntropy: 2.9949
    Accuracy: 15.60%
Epoch 5/10
    Train CrossEntropy: 3.0174 | Val CrossEntropy: 2.9912
    Accuracy: 15.20%
Epoch 6/10
    Train CrossEntropy: 2.5518 | Val CrossEntropy: 1.4669
    Accuracy: 62.80%
Epoch 7/10
    Train CrossEntropy: 1.4211 | Val CrossEntropy: 1.0535
    Accuracy: 72.70%
Epoch 8/10
    Train CrossEntropy: 1.0207 | Val CrossEntropy: 0.8319
    Accuracy: 80.50%
Epoch 9/10
    Train CrossEntropy: 0.8947 | Val CrossEntropy: 0.7857
    Accuracy: 79.80%
Epoch 10/10
    Train CrossEntropy: 0.8421 | Val CrossEntropy: 0.7071
    Accuracy: 81.20%

** Historial de entrenamiento guardado en: /home/oscar/Escritorio/TSP-

## PE Tradicional

### Justificación

Al igual que con el PE espacial, debemos hacernos las mismas preguntas antes de aplicar esta técnica: por qué y dónde.

**El por qué**

Recordemos que el PE tradicional busca representar un orden natural o jerarquía entre elementos de una secuencia. En el TSP, dado que vamos visitando las ciudades en **orden**, existe una jerarquía natural. No es lo mismo la secuencia $1 \to 2 \to 3$ que $1 \to 3 \to 2$.

Entonces, lo que buscaremos será aplicar PE directamente a la representación del **tour**, de manera que el modelo pueda entender exactamente el orden en que han sido visitadas las ciudades hasta el momento. Vale la pena recordar cómo estamos representando el tour actualmente: calculamos la media de las ciudades visitadas y concatenamos manualmente la primera y última ciudad, para luego fusionar todo en un único *embedding* global.

Como se puede ver, es información simplificada del tour. El modelo no tiene forma de saber el orden exacto en que han sido visitadas las ciudades.

**El dónde**

¿En qué parte de la arquitectura representamos el tour? El tour es información propia del **estado** del problema. Se va actualizando a medida que vamos añadiendo ciudades. Por lo tanto, es tarea del *decoder* procesar esta información. Lo conveniente es implementarlo antes del mecanismo *glimpse*, con el objetivo de armar una representación enriquecida del tour antes de utilizarlo en este mecanismo.

### Definición matemática

Para esta técnica también calculamos *embeddings* posicionales y los sumamos directamente a los *embeddings* actuales de cada ciudad. Los *embeddings* posicionales se calculan de la siguiente manera:

$$\begin{aligned} PE_{(pos, 2i)} &= \sin\left(\frac{pos}{10000^{\frac{2i}{d_{model}}}}\right) \\ PE_{(pos, 2i+1)} &= \cos\left(\frac{pos}{10000^{\frac{2i}{d_{model}}}}\right) \end{aligned}$$

La lógica es la misma que antes, solo que aquí armamos inmediatamente un embedding de dimensión $d_{model}$. La variable $pos$ representa el orden en que fue visitada una ciudad dentro del tour. La ciudad inicial tiene la posición 0, la segunda tiene la posición 1, y así sucesivamente. $i$ corresponde al índice del par de dimensiones ($i = [0, 1, ..., d_{model}/2-1]$). Al igual que antes, vamos intercalando entre $\sin$ (para las posiciones pares $2i$) y $\cos$ (para las impares $2i+1$). La representación expandida de un *embedding* posicional se vería así:

$$\begin{aligned} PE_{pos} &= \left[ \sin\left(\frac{pos}{10000^{\frac{0}{d_{model}}}}\right), \cos\left(\frac{pos}{10000^{\frac{0}{d_{model}}}}\right), \sin\left(\frac{pos}{10000^{\frac{2}{d_{model}}}}\right), \cos\left(\frac{pos}{10000^{\frac{2}{d_{model}}}}\right), ... \right] \end{aligned}$$

### Implementación y entrenamiento

En este caso, la implementación es simple, pero requiere de algunas modificaciones a la arquitectura para que tenga sentido. No podemos simplemente aplicar PE y luego promediar los *embeddings* de las ciudades visitadas para obtener el tour. Al promediar *embeddings* con información posicional, esta información se pierde o se diluye.

Por lo tanto, lo que haremos será aplicar PE seguido de una capa de *Self-Attention* entre las ciudades visitadas. Hasta el momento, las ciudades se han tratado como entidades separadas, lo que justifica la necesidad de una capa de atención que permita modelar las relaciones entre ellas y que dote al modelo de la capacidad para comprender y trazar la ruta del tour actual. Luego de esta capa, cada nodo/ciudad contiene información completa acerca de la topología del grafo y de su relación con las demás ciudades. Por consiguiente, y dado que necesitamos un único vector para representar el tour global, extraeremos directamente el *embedding* de la **última ciudad**.

La arquitectura queda de la siguiente manera:

<img src="../resources/pos_encoding.png" width="600" alt="Arquitectura con Positional Encoding Tradicional para el TSP">

La implementación completa puede revisarse en el archivo correspondiente.

In [3]:
from models.pos_encoding import TSPTransformer

# Parámetros del modelo
input_dim = 2
embed_dim = 64
num_heads = 4
num_encoder_layers = 2
num_glimpses = 2
dropout = 0.1

# Crear modelo
pe_model = TSPTransformer(
    input_dim=input_dim,
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_encoder_layers=num_encoder_layers,
    num_glimpses=num_glimpses,
    dropout_rate=dropout
)

from training.sl import sl_train, LRConfig
from training.metrics import CrossEntropyLoss, Accuracy

pe_model = sl_train(
    model=pe_model,
    epochs=10,
    train_set=train_dataset,
    val_set=val_dataset,
    batch_size=64,
    lr_config=LRConfig(value=1e-4),
    loss_fn=CrossEntropyLoss(),
    metrics=[Accuracy()],
    metrics_filename="metrics_pe.txt"
)

** Usando dispositivo: cpu

Epoch 1/10
    Train CrossEntropy: 2.2230 | Val CrossEntropy: 1.4241
    Accuracy: 65.70%
Epoch 2/10
    Train CrossEntropy: 1.3516 | Val CrossEntropy: 1.0311
    Accuracy: 77.60%
Epoch 3/10
    Train CrossEntropy: 1.0622 | Val CrossEntropy: 0.8850
    Accuracy: 79.30%
Epoch 4/10
    Train CrossEntropy: 0.9654 | Val CrossEntropy: 0.8270
    Accuracy: 80.00%
Epoch 5/10
    Train CrossEntropy: 0.8943 | Val CrossEntropy: 0.7749
    Accuracy: 80.10%
Epoch 6/10
    Train CrossEntropy: 0.8447 | Val CrossEntropy: 0.7359
    Accuracy: 81.40%
Epoch 7/10
    Train CrossEntropy: 0.8035 | Val CrossEntropy: 0.7065
    Accuracy: 80.80%
Epoch 8/10
    Train CrossEntropy: 0.7756 | Val CrossEntropy: 0.6839
    Accuracy: 81.10%
Epoch 9/10
    Train CrossEntropy: 0.7567 | Val CrossEntropy: 0.6883
    Accuracy: 79.90%
Epoch 10/10
    Train CrossEntropy: 0.7418 | Val CrossEntropy: 0.6479
    Accuracy: 81.60%

** Historial de entrenamiento guardado en: /home/oscar/Escritorio/TSP-

# PE Cíclico

## Justificación

El PE Cíclico puede entenderse como una extensión del PE Tradicional. En realidad, la justificación aquí es la misma: la necesidad del modelo de comprender el orden en que han sido visitadas las ciudades. Sin embargo, el PE Tradicional cuenta con la desventaja de que no captura la naturaleza cíclica propia del TSP, es decir, la restricción de que el *tour* debe finalizar en la misma ciudad donde comenzamos.

Esto es **crítico**. Hasta el penúltimo paso, el modelo puede haber generado un tour altamente optimizado. Sin embargo, en el último paso, el modelo se ve forzado a volver a la ciudad de origen. Aquí es donde se da cuenta de que la distancia entre la ciudad actual y la de origen es abismal, lo que arruina por completo la calidad del tour generado.

¿Qué hace entonces el PE Cíclico? Produce *embeddings* altamente similares para las primeras ciudades visitadas y las últimas por visitar. Por ejemplo, en una instancia de 50 ciudades, las ciudades en las posiciones $1, 2, 3$ y aquellas en las posiciones $47, 48, 49$ tendrán *embeddings* posicionales casi idénticos. El PE Cíclico permite que el modelo entienda que se está acercando al final del tour y que debe buscar una manera óptima de volver a la ciudad de origen.

## Definición matemática

La fórmula para el PE Cíclico es la siguiente:

$$\begin{aligned} \text{Circular-PE}(\text{pos}, 2i) &= \sin\left(\frac{\text{pos}}{10000^{2i/d_{\text{model}}}} + \frac{2\pi \cdot \text{pos}}{N}\right) \\ \text{Circular-PE}(\text{pos}, 2i+1) &= \cos\left(\frac{\text{pos}}{10000^{2i/d_{\text{model}}}} + \frac{2\pi \cdot \text{pos}}{N}\right) \end{aligned}$$

## Implementación y entrenamiento

La implementación es similar a la del PE Tradicional y puede visualizarse en el archivo correspondiente.

In [4]:
from models.circular_pe import TSPTransformer

# Parámetros del modelo
input_dim = 2
embed_dim = 64
num_heads = 4
num_encoder_layers = 2
num_glimpses = 2
dropout = 0.1

# Crear modelo
cpe_model = TSPTransformer(
    input_dim=input_dim,
    embed_dim=embed_dim,
    num_heads=num_heads,
    num_encoder_layers=num_encoder_layers,
    num_glimpses=num_glimpses,
    dropout_rate=dropout
)

from training.sl import sl_train, LRConfig
from training.metrics import CrossEntropyLoss, Accuracy

cpe_model = sl_train(
    model=cpe_model,
    epochs=10,
    train_set=train_dataset,
    val_set=val_dataset,
    batch_size=64,
    lr_config=LRConfig(value=1e-4),
    loss_fn=CrossEntropyLoss(),
    metrics=[Accuracy()],
    metrics_filename="metrics_pe.txt"
)

** Usando dispositivo: cpu

Epoch 1/10
    Train CrossEntropy: 2.1582 | Val CrossEntropy: 1.3551
    Accuracy: 67.00%
Epoch 2/10
    Train CrossEntropy: 1.3007 | Val CrossEntropy: 0.9948
    Accuracy: 76.50%
Epoch 3/10
    Train CrossEntropy: 1.0306 | Val CrossEntropy: 0.8610
    Accuracy: 79.90%
Epoch 4/10
    Train CrossEntropy: 0.9201 | Val CrossEntropy: 0.7975
    Accuracy: 80.50%
Epoch 5/10
    Train CrossEntropy: 0.8753 | Val CrossEntropy: 0.7536
    Accuracy: 80.60%
Epoch 6/10
    Train CrossEntropy: 0.8232 | Val CrossEntropy: 0.7185
    Accuracy: 81.60%
Epoch 7/10
    Train CrossEntropy: 0.7843 | Val CrossEntropy: 0.6959
    Accuracy: 81.00%
Epoch 8/10
    Train CrossEntropy: 0.7620 | Val CrossEntropy: 0.6783
    Accuracy: 81.40%
Epoch 9/10
    Train CrossEntropy: 0.7277 | Val CrossEntropy: 0.6659
    Accuracy: 82.60%
Epoch 10/10
    Train CrossEntropy: 0.7274 | Val CrossEntropy: 0.6401
    Accuracy: 82.90%

** Historial de entrenamiento guardado en: /home/oscar/Escritorio/TSP-

## Validación

Podemos comparar los diferentes enfoques para ver cuál entrega mejores resultados:

In [5]:
from solvers.eval import evaluate
from data.adapters.input.default import DefaultInputAdapter

# Configuración compartida
input_config = (DefaultInputAdapter, 50)
instance_file = "benchmarks/B50.pkl"

# Agrupamos los modelos en un diccionario para evaluar secuencialmente
models_to_evaluate = {
    "Positional Encoding Espacial (SPE)": spe_model,
    "Positional Encoding Tradicional (PE)": pe_model,
    "Positional Encoding Cíclico (CPE)": cpe_model
}

# Diccionario para almacenar las soluciones sin sobrescribirlas
evaluation_results = {}

for name, model in models_to_evaluate.items():
    print(f"\n{'='*50}")
    print(f"Evaluando: {name}")
    print(f"{'='*50}")
    
    model_sols, ort_sols = evaluate(
        model=model,
        instance_file=instance_file,
        input_adapter_config=input_config,
        num_workers=None  # Utiliza todos los núcleos de CPU disponibles
    )
    
    # Guardamos los resultados por si se necesitan analizar o graficar posteriormente
    evaluation_results[name] = {
        "model_sols": model_sols,
        "ort_sols": ort_sols
    }


Evaluando: Positional Encoding Espacial (SPE)
Iniciando evaluación conjunta de 100 instancias con 12 workers...

RESULTADOS DE LA VALIDACIÓN
Costo promedio Modelo:   6.69
Costo promedio OR-Tools: 5.82
Gap de optimalidad:      14.94% ± 7.13%


Evaluando: Positional Encoding Tradicional (PE)
Iniciando evaluación conjunta de 100 instancias con 12 workers...

RESULTADOS DE LA VALIDACIÓN
Costo promedio Modelo:   6.69
Costo promedio OR-Tools: 5.82
Gap de optimalidad:      15.00% ± 6.15%


Evaluando: Positional Encoding Cíclico (CPE)
Iniciando evaluación conjunta de 100 instancias con 12 workers...

RESULTADOS DE LA VALIDACIÓN
Costo promedio Modelo:   6.68
Costo promedio OR-Tools: 5.82
Gap de optimalidad:      14.77% ± 6.43%

